# FerrisRes: Gemma-4 Distillation on Colab T4

Distill Gemma-4 models into FerrisRes Block AttnRes format.

**Supported models (T4 / 16GB VRAM):**
- **E2B** (~4.6 GB FP16, 35 layers, dense) ✅
- **E4B** (~9.0 GB FP16, 42 layers, dense) ✅

**Too large for T4 (requires A100 80GB):**
- **26B-A4B** (~52 GB FP16, MoE-128) ❌
- **31B** (~62 GB FP16, dense) ❌

FerrisRes uses profile-driven dispatch — it auto-detects the T4, plans memory, and tiles
large matmuls to prevent OOM. No `--gpu` flag needed.

In [ ]:
#@title Configuration
model_config = "e2b" #@param ["e2b", "e4b"]
learning_rate = 0.00004 #@param {type:"number"}
seq_len = 256 #@param {type:"integer"}
max_steps = 10000 #@param {type:"integer"}
converge_threshold = 0.001 #@param {type:"number"}
converge_patience = 150 #@param {type:"integer"}

print(f"Config: {model_config}, seq_len={seq_len}, lr={learning_rate}, max_steps={max_steps}")

In [ ]:
# 1. Install Vulkan ICD + Rust
!apt-get update -qq

# Detect NVIDIA driver version and install matching Vulkan ICD
import subprocess
try:
    driver_version = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]
    ).decode("utf-8").strip()
    major = driver_version.split('.')[0]
    print(f"NVIDIA driver {driver_version}, installing libnvidia-gl-{major}...")
    !apt-get install -y -qq libvulkan1 vulkan-tools libnvidia-gl-{major}
except Exception as e:
    print(f"Driver detection failed ({e}), installing generic Vulkan...")
    !apt-get install -y -qq libvulkan1 vulkan-tools

# Verify T4 visible via Vulkan
!echo '=== Vulkan GPU Detection ===' && vulkaninfo --summary 2>/dev/null | grep -E "deviceName|apiVersion" | head -5

# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -1
import os
os.environ['PATH'] += ":/root/.cargo/bin"
!rustc --version

In [ ]:
# 2. Clone FerrisRes + Download Model Weights + Tokenizer
!git clone https://github.com/shift/FerrisRes.git
%cd FerrisRes

hf_model_id = f"google/gemma-4-{model_config}-it"
print(f"Downloading {hf_model_id}...")

!wget -q --show-progress https://huggingface.co/{hf_model_id}/resolve/main/model.safetensors
!wget -q https://huggingface.co/{hf_model_id}/resolve/main/tokenizer.json

# Training data: WikiText-2 (~2MB, ~50K tokens)
!wget -q https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt -O training_data.txt

!ls -lh model.safetensors tokenizer.json training_data.txt

In [ ]:
# 3. Check Hardware Profile (should detect T4)
!cargo run --release -- info

In [ ]:
# 4. Run Distillation
# Profile-driven dispatch auto-selects GPU matmuls.
# LM head is auto-tiled to fit T4 max_buffer_size.
!cargo run --release -- distill \
  --model-path ./model.safetensors \
  --config {model_config} \
  --seq-len {seq_len} \
  --steps {max_steps} \
  --learning-rate {learning_rate} \
  --temperature 2.0 \
  --tokenizer ./tokenizer.json \
  --data training_data.txt \
  --output distilled_{model_config} \
  --log-every 50 \
  --converge {converge_threshold} \
  --converge-patience {converge_patience}

In [ ]:
# 5. (Optional) Resume if Colab session disconnected
# Uncomment and re-run to continue from last checkpoint:
# !cargo run --release -- distill \
#   --model-path ./model.safetensors \
#   --config {model_config} \
#   --seq-len {seq_len} \
#   --steps {max_steps} \
#   --learning-rate {learning_rate} \
#   --temperature 2.0 \
#   --tokenizer ./tokenizer.json \
#   --data training_data.txt \
#   --output distilled_{model_config} \
#   --resume distilled_{model_config}.bin.checkpoint \
#   --log-every 50 \
#   --converge {converge_threshold} \
#   --converge-patience {converge_patience}

In [ ]:
# 6. Visualize Distillation Progress
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

loss_file = f'distilled_{model_config}.bin.loss_curve.csv'
try:
    df = pd.read_csv(loss_file)
    df = df.replace('n/a', np.nan)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

    ax1.plot(df['step'], df['kl_loss'], linewidth=2)
    ax1.set_ylabel('KL Divergence Loss')
    ax1.set_title(f'Gemma 4 {model_config.upper()} Distillation')
    ax1.grid(True, linestyle='--', alpha=0.6)

    if 'cosine_sim_avg' in df.columns:
        valid_cos = df.dropna(subset=['cosine_sim_avg'])
        valid_cos['cosine_sim_avg'] = pd.to_numeric(valid_cos['cosine_sim_avg'])
        ax2.plot(valid_cos['step'], valid_cos['cosine_sim_avg'],
                 color='darkorange', marker='o', markersize=3, alpha=0.8)
        ax2.set_ylabel('Cosine Similarity (Teacher vs Student)')
        ax2.set_xlabel('Step')
        ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print(f'{loss_file} not found. Did the distillation complete?')